In [1]:
#Java Environment Setup
import os
import sys
import glob

venv_path = sys.prefix
jvm_dir = os.path.join(venv_path, "jvm")
java_execs = glob.glob(os.path.join(jvm_dir, "**/bin/java"), recursive=True)

if java_execs:
    resolved_java_home = os.path.dirname(os.path.dirname(os.path.abspath(java_execs[0])))
    os.environ["JAVA_HOME"] = resolved_java_home
    os.environ["PATH"] = os.path.join(resolved_java_home, "bin") + os.pathsep + os.environ.get("PATH", "")
    print(f"JAVA_HOME configured at: {resolved_java_home}")
else:
    print("Warning: JDK binary not found in .venv/jvm")

JAVA_HOME configured at: /home/wnder/Documents/repos/teleSpikeRedo/.venv/jvm/jdk-17.0.20+8


In [2]:
#setup and cofigs
import json
import re
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, explode, udf, to_json, struct
from pyspark.sql.types import StructType, StructField, StringType, LongType, ArrayType

with open("config.json", "r", encoding="utf-8") as f:
    config = json.load(f)

BOOTSTRAP_SERVER = config["kafka"]["bootstrap_server"]
RAW_TOPIC = config["kafka"]["raw_topic"]
TOKENS_TOPIC = config["kafka"]["tokens_topic"]
FOLLOWED_TOKENS = config["followed_tokens"]

spark = SparkSession.builder \
    .appName("TelegramTokenizer") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print("Spark session initialized.")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/30 11:06:12 WARN Utils: Your hostname, debian, resolves to a loopback address: 127.0.1.1; using 10.100.102.22 instead (on interface enp0s31f6)
26/08/30 11:06:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/wnder/Documents/repos/teleSpikeRedo/.venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/wnder/.ivy2.5.2/cache
The jars for the packages stored in: /home/wnder/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e8210ece-ff5d-44df-89c9-100ce87b9967;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9

Spark session initialized.


In [3]:
#topic creation, in case they werent created yet
from kafka import KafkaAdminClient
from kafka.admin import NewTopic
from kafka.errors import TopicAlreadyExistsError

admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP_SERVER)
topics = [
    NewTopic(name=RAW_TOPIC, num_partitions=1, replication_factor=1),
    NewTopic(name=TOKENS_TOPIC, num_partitions=4, replication_factor=1)
]

for t in topics:
    try:
        admin.create_topics([t], validate_only=False)
        print(f"Topic '{t.name}' ready.")
    except TopicAlreadyExistsError:
        print(f"Topic '{t.name}' already exists.")
admin.close()

Topic 'telegram-raw' ready.
Topic 'telegram-tokens' ready.


In [4]:
# Raw JSON schema as produced by the Telethon producer
raw_schema = StructType([
    StructField("channel", StringType(), True),
    StructField("text", StringType(), True),
    StructField("ts", LongType(), True)
])

# a simple worker functaion for parsing one message into tokes
def tokenize(text):
    if not text:
        return []
    #Hebrew and alphanumeric words of length >= 2
    return re.findall(r"[\u0590-\u05fe\w]{2,}", text.lower())

#turn to a udf
tokenize_udf = udf(tokenize, ArrayType(StringType()))

/home/wnder/Documents/repos/teleSpikeRedo/.venv/lib/python3.13/site-packages/pyspark/sql/udf.py:116: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [5]:
# connect to kafka topic
raw_stream_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVER) \
    .option("subscribe", RAW_TOPIC) \
    .option("startingOffsets", "latest") \
    .option("failOnDataLoss", "false") \
    .load()

# parse data into json and then into data frame
parsed_df = raw_stream_df.select(
    from_json(col("value").cast("string"), raw_schema).alias("data")
).select("data.*")

#tokenize, explode, and filter, in parallel
tokens_df = parsed_df \
    .withColumn("word", explode(tokenize_udf(col("text")))) \
    .filter(col("word").isin(FOLLOWED_TOKENS))

#parse into the kafka token stream format:: key: token, value: timestamp
kafka_output_df = tokens_df.select(
    col("word").alias("key"),
    to_json(struct(col("ts"))).alias("value")
)

#stream to kafka secend producer
query = kafka_output_df.writeStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVER) \
    .option("topic", TOKENS_TOPIC) \
    .option("checkpointLocation", "/tmp/spark-kafka-tokenizer-checkpoint") \
    .outputMode("append") \
    .start()
print("This spark session is live and ready, awating termination")
query.awaitTermination()

26/08/30 11:06:19 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


This spark session is live and ready, awating termination


ERROR:root:KeyboardInterrupt while sending command.                             
Traceback (most recent call last):
  File "/home/wnder/Documents/repos/teleSpikeRedo/.venv/lib/python3.13/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/home/wnder/Documents/repos/teleSpikeRedo/.venv/lib/python3.13/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ~~~~~~~~~~~~~~~~~~~~^^
  File "/usr/lib/python3.13/socket.py", line 719, in readinto
    return self._sock.recv_into(b)
           ~~~~~~~~~~~~~~~~~~~~^^^
KeyboardInterrupt


KeyboardInterrupt: 